# Graficas para el informe

## Cargamos los datos

In [1]:
import pandas as pd
import numpy as np

dataset = pd.read_csv("../data/clean-data/La_Laguna/Tabla.csv", sep=";")
dataset.info()

# Filtrar las filas donde Valor no sea '???'
dataset = dataset[dataset['Valor'] != '???']

dataset.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2952 entries, 0 to 2951
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Anio     2952 non-null   object
 1   Valor    2952 non-null   object
 2   Carrera  2952 non-null   object
 3   Genero   2952 non-null   object
 4   Tasa     2952 non-null   object
dtypes: object(5)
memory usage: 115.4+ KB


,Anio,Valor,Carrera,Genero,Tasa
8,2022-2023,75.1,Todos los ámbitos,Ambos Sexos,Rendimiento
9,2021-2022,72.18,Todos los ámbitos,Ambos Sexos,Rendimiento
10,2020-2021,77.06,Todos los ámbitos,Ambos Sexos,Rendimiento
11,2019-2020,81.09,Todos los ámbitos,Ambos Sexos,Rendimiento
12,2018-2019,72.57,Todos los ámbitos,Ambos Sexos,Rendimiento


## Tasa de éxito

### Calculamos percentiles

In [12]:
# Filtramos solo los datos de tasa de éxito
tasa_exito = dataset[dataset['Tasa'] == 'Exito'].copy()

# Convertimos el valor a numérico
tasa_exito['Valor'] = pd.to_numeric(tasa_exito['Valor'], errors='coerce')

# Calculamos los percentiles por año
percentiles_por_año = tasa_exito.groupby('Anio')['Valor'].quantile([0.25, 0.50, 0.75]).unstack()
percentiles_por_año = percentiles_por_año.round(2)
percentiles_por_año.columns = ['P_25', 'P_50', 'P_75']
percentiles_por_año_exito = percentiles_por_año.reset_index()

print(percentiles_por_año_exito)

        Anio   P_25   P_50   P_75
0  2015-2016  82.60  87.17  90.15
1  2016-2017  82.50  85.84  90.58
2  2017-2018  82.24  86.37  91.05
3  2018-2019  82.33  87.04  91.32
4  2019-2020  87.42  92.01  94.36
5  2020-2021  83.78  88.89  92.96
6  2021-2022  80.10  86.61  92.42
7  2022-2023  82.53  86.49  92.12


### Evolución en la tasa de éxito por cohorte de egreso

In [3]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=percentiles_por_año_exito['Anio'], y=percentiles_por_año_exito['P_25'],
    mode='lines', name='P25 del campo',
    line=dict(color='red', width=2)
))

fig.add_trace(go.Scatter(
    x=percentiles_por_año_exito['Anio'], y=percentiles_por_año_exito['P_50'],
    mode='lines', name='P50 del campo',
    line=dict(color='black', width=2, dash='dot')
))

fig.add_trace(go.Scatter(
    x=percentiles_por_año_exito['Anio'], y=percentiles_por_año_exito['P_75'],
    mode='lines', name='P75 del campo',
    line=dict(color='green', width=2)
))

dataset_informatica = dataset[(dataset['Carrera'] == 'Informática') & (dataset['Genero'] == 'Ambos Sexos') & (dataset['Tasa'] == 'Exito')].copy()
# print(dataset_informatica)

fig.add_trace(go.Scatter(
    x=dataset_informatica['Anio'], 
    y=dataset_informatica['Valor'], # Ejemplo visual
    mode='lines+markers', name='Titulación',
    line=dict(color='purple', width=4),
    marker=dict(symbol='diamond', size=10)
))

fig.update_layout(
    title="Tasa de éxito al egreso por cohorte",
    xaxis_title="Cohorte de Egreso",
    yaxis_title="Tasa de éxito al egreso",
    yaxis=dict(
        ticksuffix="%",      # Agrega el símbolo % al eje Y
        gridcolor='lightgrey'
    ),
    plot_bgcolor='white', 
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5)
)



fig.show()

### Evolución en la tasa de éxito por cohorte de egreso

In [ ]:
dataset_all_data = dataset[(dataset['Carrera'] == 'Informática') & (dataset['Genero'] == 'Ambos Sexos') & (dataset['Tasa'] == 'Exito')].copy()
dataset_all_data = pd.merge(percentiles_por_año_exito, dataset_all_data, on='Anio')
dataset_all_data = dataset_all_data[['Anio', 'Valor', 'P_25', 'P_50', 'P_75']]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>Año</b>", "<b>Tasa de Éxito (%)</b>", "<b>P25 (%)</b>", "<b>P50 (%)</b>", "<b>P75 (%)</b>"],
        fill_color='#4B1E78', # Morado oscuro similar al de tu imagen
        align='center',
        font=dict(color='white', size=12),
        line_color='black',
        height=35
    ),
    cells=dict(
        values=[dataset_all_data[col] for col in dataset_all_data.columns],
        fill_color='white',
        align=['center'],
        font=dict(color='black', size=12),
        line_color='black',
        height=30
    )
)])

fig.update_layout(
    title="<b>Evolución en la tasa de éxito por cohorte de egreso</b>",
    margin=dict(l=10, r=10, t=60, b=10)
)

fig.show()


## Tasa de evaluación

## Tasa de rendimiento